# Mini-projet : Chatbot PDF Gemini
Dernière mise à jour : 13 mai 2025

Introduction
Dans ce mini-projet, vous créerez un chatbot capable de lire et de répondre à des questions sur des documents PDF grâce à l'IA conversationnelle de Google Gemini et à la recherche vectorielle avec LangChain et FAISS. Ce chatbot permet aux utilisateurs de télécharger plusieurs PDF, de traiter leur contenu en intégrations et de poser des questions via une interface Streamlit.



👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Comment extraire et fragmenter du texte à partir de fichiers PDF.
Comment générer et stocker des intégrations de texte à l'aide de Google Gemini.
Comment créer un magasin vectoriel à l'aide de FAISS.
Comment créer une chaîne d'assurance qualité de récupération conversationnelle avec LangChain.
Comment déployer une interface de chat interactive à l'aide de Streamlit.
Comment collecter les informations de contact des utilisateurs via la saisie de formulaire.


🛠️ Ce que vous allez créer
Vous créerez une application Web de chatbot intelligente où les utilisateurs pourront télécharger des fichiers PDF, traiter le contenu et interagir avec un chatbot qui répond aux questions en fonction des documents téléchargés.

# **Analyse approfondie** du projet 

et une **procédure directe, efficace et professionnelle** pour le résoudre **sans s’enliser dans les détails inutiles de l’énoncé**, tout en garantissant un résultat fonctionnel, clair et réutilisable.

---

## Analyse stratégique

Ce projet consiste à **créer une application web** qui permet :

* Le dépôt de fichiers PDF
* L’extraction et le découpage du texte
* L’indexation vectorielle via embeddings Gemini + FAISS
* L’interrogation par chat du contenu PDF (RAG) avec Gemini
* La collecte d’infos utilisateur via formulaire

Les points **critiques** sont :

1. Extraction et découpage des PDF (gestion multi-fichiers, robustesse)
2. Création/chargement du store FAISS, gestion efficace du rechargement
3. Construction d’une chaîne QA conversationnelle (prompt + historique minimal)
4. Intégration propre dans Streamlit, avec gestion des fichiers et du contexte session utilisateur
5. Collecte, stockage et robustesse de la sauvegarde CSV

**Ce qui compte** :

* Robustesse de l’upload / extraction PDF
* Utilisation correcte des embeddings Gemini (GoogleGenerativeAIEmbeddings)
* Intégration FAISS (création, update, load)
* Prompt de QA bien rédigé, fallback si pas d’info trouvée
* Simplicité de l’UI Streamlit (uploads, chat, formulaire)

---

## Procédure optimale de résolution

### 1. **Préparer l’environnement**

* Créer un dossier de projet dédié (ex : `chatbot_pdf_gemini`)
* Copier le `requirements.txt` fourni, installer avec `pip install -r requirements.txt`
* Créer un `.env` avec la clé API Google (ex : `GOOGLE_API_KEY=xxxx`)
* Créer une structure de base :

  ```
  chatbot_pdf_gemini/
  ├── main.py
  ├── requirements.txt
  ├── .env
  ├── user_info.csv (sera généré)
  ```

### 2. **Développer les modules fonctionnels indépendants**

#### a. **Extraction du texte des PDF**

* Créer une fonction robuste utilisant PyPDF2 pour extraire tout le texte de tous les fichiers PDF uploadés.
* Gérer les fichiers multiples et l’unicité des pages (erreurs, fichiers corrompus).

```python
def get_pdf_text(pdf_docs):
    from PyPDF2 import PdfReader
    text = ""
    for pdf in pdf_docs:
        reader = PdfReader(pdf)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text
```

#### b. **Découpage en chunks**

* Utiliser LangChain `RecursiveCharacterTextSplitter` avec taille et overlap adaptés.
* Gérer la langue (français/anglais) si besoin, mais par défaut : découpe simple.

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

def get_text_chunks(text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    return splitter.split_text(text)
```

#### c. **Génération des embeddings et création FAISS**

* Utiliser les embeddings Gemini via `langchain_google_genai.GoogleGenerativeAIEmbeddings`.
* Sauvegarder l’index FAISS localement, le recharger si déjà existant.
* Prévoir une fonction d’initialisation + update (pour éviter les collisions/duplications).

```python
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores.faiss import FAISS

def get_vector_store(chunks):
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    vector_store = FAISS.from_texts(chunks, embedding=embeddings)
    vector_store.save_local("faiss_index")
```

#### d. **Chaîne de récupération conversationnelle (RAG) avec Gemini**

* Prompt explicite : *“Réponds uniquement si l’info est dans le contexte, sinon dis ‘la réponse n’est pas disponible dans le contexte’.”*
* Utiliser la classe adaptée pour l’appel LLM Gemini.

```python
from langchain.chains.question_answering import load_qa_chain
from langchain.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

def get_conversational_chain():
    prompt_template = """
    Utilise uniquement les informations suivantes pour répondre à la question.
    Si la réponse n’est pas disponible dans le contexte, dis « la réponse n’est pas disponible dans le contexte ».
    Contexte : {context}
    Question : {question}
    Réponse :
    """
    model = ChatGoogleGenerativeAI(model="gemini-1.5-pro-latest")
    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
    chain = load_qa_chain(llm=model, chain_type="stuff", prompt=prompt)
    return chain
```

#### e. **Recherche vectorielle + génération de réponse**

* Charger FAISS à la volée, utiliser les embeddings, chercher les docs les plus proches.

```python
def user_input(user_question):
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    docs = db.similarity_search(user_question)
    chain = get_conversational_chain()
    context = "\n".join([doc.page_content for doc in docs])
    response = chain(
        {"input_documents": docs, "context": context, "question": user_question},
        return_only_outputs=True)
    return response['output_text']
```

#### f. **Gestion de l’historique et du reset**

```python
def clear_chat_history():
    import streamlit as st
    st.session_state.messages = [{"role": "assistant", "content": "Upload some PDFs and ask me a question."}]
```

#### g. **Collecte d’infos utilisateur (CSV)**

```python
import csv, os

def save_user_info(name, phone, email):
    file_exists = os.path.isfile('user_info.csv')
    with open('user_info.csv', mode='a', newline='', encoding='utf-8') as file:
        fieldnames = ['Name', 'Phone', 'Email']
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({'Name': name, 'Phone': phone, 'Email': email})
```

---

### 3. **Intégration Streamlit simple et directe**

* Upload de PDF (sidebar)
* Affichage chat (zone principale)
* Formulaire infos (pop-up ou zone conditionnelle)

#### **Main Streamlit flow :**

```python
import streamlit as st

def main():
    st.set_page_config(page_title="PDF Gemini Chatbot", page_icon=":robot_face:", layout="wide")
    with st.sidebar:
        st.title("Menu")
        pdf_docs = st.file_uploader(
            "Upload your PDF Files and Click on the Submit & Process Button", accept_multiple_files=True)
        if st.button("Submit & Process"):
            with st.spinner("Processing PDFs..."):
                raw_text = get_pdf_text(pdf_docs)
                text_chunks = get_text_chunks(raw_text)
                get_vector_store(text_chunks)
                st.success("Done")

    st.title("Chat with PDF files using Gemini")
    st.write("Welcome to the chat!")

    if "messages" not in st.session_state:
        st.session_state.messages = [{"role": "assistant", "content": "Upload some PDFs and ask me a question."}]

    if st.button("Clear Chat History"):
        clear_chat_history()

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(f"**{message['role'].capitalize()}:** {message['content']}")

    prompt = st.chat_input("Posez une question sur vos PDF")
    if prompt:
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(f"**User:** {prompt}")
        if "call me" in prompt.lower():
            st.session_state.collecting_info = True
        if not st.session_state.messages[-1]["role"] == "assistant":
            with st.chat_message("assistant"):
                with st.spinner("Gemini réfléchit..."):
                    response = user_input(prompt)
                    st.session_state.messages.append({"role": "assistant", "content": response})
                    st.markdown(f"**Assistant:** {response}")

    # Gestion formulaire contact
    if "collecting_info" in st.session_state and st.session_state.collecting_info:
        st.subheader("Entrez vos coordonnées pour être contacté :")
        with st.form(key="contact_form"):
            name = st.text_input("Nom")
            phone = st.text_input("Téléphone")
            email = st.text_input("E-mail")
            submit_button = st.form_submit_button("Envoyer")
            if submit_button:
                save_user_info(name, phone, email)
                st.session_state.messages.append(
                    {"role": "assistant", "content": f"Merci, {name}. Nous vous contacterons au {phone} ou {email}."})
                st.session_state.collecting_info = False

if __name__ == "__main__":
    main()
```

---

## **Résumé** de la méthode et **points-clés**

* **Priorité à la robustesse** sur l’extraction PDF et la gestion FAISS.
* **Pas d’artifices** : ne jamais multiplier les fonctions/méthodes inutiles, rester strictement orienté utilité.
* **Stockage local** des embeddings (FAISS index) et des infos utilisateurs (CSV).
* **Utilisation explicite** des modules clés (Streamlit, FAISS, Gemini Embeddings, PyPDF2).
* **Interface Streamlit** minimaliste mais complète : upload, chat, formulaire.
* **Prompt LLM** explicite, avec fallback automatique si hors contexte.
* **Réutilisable** et extensible pour tout autre usage documentaire.

---

### **Conclusion**

Avec cette approche, tu obtiens **une solution directe, lisible, modulaire et réutilisable**, sans surcomplexifier, tout en couvrant toutes les exigences pratiques pour un vrai chatbot PDF RAG moderne.
Si tu veux le code complet, dis-le, je te fournis un fichier prêt à l’emploi.


# Realisation

Voici **toute l’arborescence** et le **contenu complet de chaque fichier** pour ce projet.
---

# Arborescence

```
chatbot_pdf_gemini/
├── main.py
├── requirements.txt
├── .env
├── user_info.csv   # (ce fichier sera créé à la première sauvegarde utilisateur)
```

---

## 1. **requirements.txt**

```txt
streamlit
google-generativeai
python-dotenv
langchain
PyPDF2
chromadb
faiss-cpu
langchain_google_genai
langchain-community
```

---

## 2. **.env**

> Place ta clé API Google dans ce fichier, **remplace** `TA_CLE_API_ICI` par ta vraie clé.

```env
GOOGLE_API_KEY=TA_CLE_API_ICI
```

---

## 3. **main.py**

```python
import os
import csv
import streamlit as st
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.vectorstores.faiss import FAISS
from langchain.chains.question_answering import load_qa_chain
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()

# Extraction du texte des PDF
def get_pdf_text(pdf_docs):
    text = ""
    for pdf in pdf_docs:
        reader = PdfReader(pdf)
        for page in reader.pages:
            t = page.extract_text()
            if t:
                text += t
    return text

# Découpage en morceaux
def get_text_chunks(text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    return splitter.split_text(text)

# Génération des embeddings et index FAISS
def get_vector_store(chunks):
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    vector_store = FAISS.from_texts(chunks, embedding=embeddings)
    vector_store.save_local("faiss_index")

# Chaîne de QA avec prompt explicite
def get_conversational_chain():
    prompt_template = """
    Utilise uniquement les informations suivantes pour répondre à la question.
    Si la réponse n’est pas disponible dans le contexte, dis « la réponse n’est pas disponible dans le contexte ».
    Contexte : {context}
    Question : {question}
    Réponse :
    """
    model = ChatGoogleGenerativeAI(model="gemini-1.5-pro-latest")
    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
    chain = load_qa_chain(llm=model, chain_type="stuff", prompt=prompt)
    return chain

# Recherche et réponse du bot
def user_input(user_question):
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    docs = db.similarity_search(user_question)
    chain = get_conversational_chain()
    context = "\n".join([doc.page_content for doc in docs])
    response = chain(
        {"input_documents": docs, "context": context, "question": user_question},
        return_only_outputs=True)
    return response['output_text']

# Reset de l'historique chat
def clear_chat_history():
    st.session_state.messages = [{"role": "assistant", "content": "Upload some PDFs and ask me a question."}]

# Sauvegarde infos utilisateur
def save_user_info(name, phone, email):
    file_exists = os.path.isfile('user_info.csv')
    with open('user_info.csv', mode='a', newline='', encoding='utf-8') as file:
        fieldnames = ['Name', 'Phone', 'Email']
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({'Name': name, 'Phone': phone, 'Email': email})

def main():
    st.set_page_config(page_title="PDF Gemini Chatbot", page_icon=":robot_face:", layout="wide")
    with st.sidebar:
        st.title("Menu")
        pdf_docs = st.file_uploader(
            "Upload your PDF Files and Click on the Submit & Process Button", accept_multiple_files=True)
        if st.button("Submit & Process"):
            with st.spinner("Processing PDFs..."):
                raw_text = get_pdf_text(pdf_docs)
                text_chunks = get_text_chunks(raw_text)
                get_vector_store(text_chunks)
                st.success("Done")

    st.title("Chat with PDF files using Gemini")
    st.write("Welcome to the chat!")

    if "messages" not in st.session_state:
        st.session_state.messages = [{"role": "assistant", "content": "Upload some PDFs and ask me a question."}]

    if st.button("Clear Chat History"):
        clear_chat_history()

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(f"**{message['role'].capitalize()}:** {message['content']}")

    prompt = st.chat_input("Posez une question sur vos PDF")
    if prompt:
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(f"**User:** {prompt}")
        if "call me" in prompt.lower():
            st.session_state.collecting_info = True
        if not st.session_state.messages[-1]["role"] == "assistant":
            with st.chat_message("assistant"):
                with st.spinner("Gemini réfléchit..."):
                    response = user_input(prompt)
                    st.session_state.messages.append({"role": "assistant", "content": response})
                    st.markdown(f"**Assistant:** {response}")

    if "collecting_info" in st.session_state and st.session_state.collecting_info:
        st.subheader("Entrez vos coordonnées pour être contacté :")
        with st.form(key="contact_form"):
            name = st.text_input("Nom")
            phone = st.text_input("Téléphone")
            email = st.text_input("E-mail")
            submit_button = st.form_submit_button("Envoyer")
            if submit_button:
                save_user_info(name, phone, email)
                st.session_state.messages.append(
                    {"role": "assistant", "content": f"Merci, {name}. Nous vous contacterons au {phone} ou {email}."})
                st.session_state.collecting_info = False

if __name__ == "__main__":
    main()
```

---

## 4. **user\_info.csv** (optionnel)

Ce fichier **n’a pas à être créé manuellement**. Il sera généré lors du premier enregistrement utilisateur via le formulaire.

---

### **Utilisation**

1. **Place ta clé dans `.env`**.
2. **Ouvre un terminal dans le dossier**.
3. **Installe les dépendances** :

   ```
   pip install -r requirements.txt
   ```
4. **Lance l’application** :

   ```
   streamlit run main.py
   ```
5. **Utilise l’interface dans ton navigateur** :
   Upload PDF(s), pose des questions, vérifie la création de `user_info.csv` après un formulaire.

---
